# Motion denoising: how the settings were optimized

The settings of RSGM in DPoser's optimization (score distillation on SO(3)^J) were tuned on the validation split of PoseD-Flow's protocol (`hp/evaluation/denoising.py`: 100 chunks of HDM05, SFU, MoSh) by Optuna studies (`scripts/tune_denoising.py`, `config/denoising_tuning/`), one per noise level and model:

| study | model | trials |
|---|---|---|
| `rsgm_sds_40mm`, `rsgm_sds_100mm` | RSGM 512x4 c10 (isotropic) | 40 each |
| `rsgm_sds_aniso_40mm`, `rsgm_sds_aniso_100mm` | RSGM abl_residual_aniso c10 (per-joint noise) | 30 each |

**The choice is one number**: the lowest validation MPJPE (the objective of the studies; the rule written in `rsgm_sds_<mm>mm.yaml` before the test). The best trial of `rsgm_sds_<mm>mm` gave the settings of every isotropic RSGM row at that noise level; "ours" is the RSGM row with the lowest validation MPJPE at these settings. MPVPE and the cost played no part.

This notebook shows how the studies went (sections 2 and 3) and where all the settings tried so far lie in MPJPE, MPVPE and cost, next to DPoser and PoseD-Flow (section 4). The chosen points are marked with a black star. Nothing is computed: the records are read from the caches of `scripts/evaluate_denoising.py` and from the study files, by `hp.evaluation.denoising_results`.

In [ ]:
import sys
from pathlib import Path

# the code of the checkout this notebook is in (e.g. a worktree), before an editable install of another one
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
for path in (ROOT / "src", ROOT / "scripts", ROOT / "explore" / "evaluation" / "denoising_aniso"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display

import evaluate_denoising as ed
import rows as aniso_rows  # noqa: F401  (adds the rows of the per-joint-noise models to ed.MODELS)
from hp.evaluation import denoising_results as R
from hp.evaluation.denoising import PAPER, SIGMAS

assert R.__file__.startswith(str(ROOT / "src")), f"hp comes from {R.__file__}: restart the kernel"

# ---- parameters ----------------------------------------------------------------------
RESULTS = Path("~/data/experiments/denoising").expanduser()
# the Mac cache first, then copies from the servers (cache_<server>, e.g. the validation rows run there)
CACHES = [RESULTS / "cache"] + sorted(p for p in RESULTS.glob("cache_*") if p.is_dir())
STUDIES = sorted((RESULTS / "optuna").glob("*.db"))   # copies from the servers too
CHOSEN_STUDY = {0.04: "rsgm_sds_40mm", 0.1: "rsgm_sds_100mm"}  # their best trial set the settings of all RSGM rows
OURS = "RSGM 1536x6: their c10, DPoser method"        # "ours" by the rule
REFERENCES = {"DPoser: their full, DPoser method": "DPoser", "PoseD-Flow: their checkpoint": "PoseD-Flow"}
SHOW_OTHER = False                                    # rows cached with other settings than their current spec
FAMILIES = {                                          # colour of each family, in the order of the legend
    "RSGM (ours)": "#d62728",
    "RSGM, per-joint noise": "#ff9896",
    "DPoser": "#1f77b4",
    "PoseD-Flow": "#2ca02c",
    "w/o prior": "#7f7f7f",
}
MARKERS = {"dposer": "o", "dflow": "s"}
ZOOM = 1.1   # the scatter plots show the points up to ZOOM x the error of "w/o prior"
SAVE_DIR = "~/data/denoising_figures"
# ----------------------------------------------------------------------------------------


def family_of(record):
    model = record["model"]
    if model == "none":
        return "w/o prior"
    if model.endswith(".ckpt"):
        return "PoseD-Flow"
    if "dposer" in model:
        return "DPoser"
    return "RSGM, per-joint noise" if "aniso" in model else "RSGM (ours)"


records = R.load(CACHES, STUDIES, models=ed.MODELS)
records = [r for r in records if SHOW_OTHER or not r["label"].endswith("[other settings]")]
for r in records:
    r["family"] = family_of(r)
    r["nfe_plot"] = max(r["nfe"], 1)
    r["chosen"] = r["label"] == OURS
for sigma, study in CHOSEN_STUDY.items():
    trials = [r for r in records if r.get("study") == study]
    if trials:
        min(trials, key=lambda r: r["mpjpe"])["chosen"] = True
studies = {}
for r in records:
    if "study" in r:
        studies.setdefault(r["study"], []).append(r)
for trials in studies.values():
    trials.sort(key=lambda r: r["trial"])
print(f"{len(records)} records: {sum('study' in r for r in records)} trials of {len(studies)} studies, "
      f"{sum('study' not in r for r in records)} cached rows ({', '.join(p.name for p in CACHES)})")

## 1. The studies and their best trials

Per study: the search space (from its config), the number of trials, and the five best trials by validation MPJPE (the objective), with their MPVPE and settings. `*` marks the trial whose settings were taken.

In [ ]:
KEYS = ("t_max", "t_min_fraction", "sds_weight", "lr", "steps_per_iter")


def fmt(v):
    return f"{v:.3g}" if isinstance(v, float) else str(v)


for name, trials in sorted(studies.items(), key=lambda kv: (kv[1][0]["sigma"], kv[0])):
    first = trials[0]
    print(f"\n{name}: {first['model']}, {round(first['sigma'] * 1000)} mm, {len(trials)} trials on "
          f"{first['sequences']} validation sequences")
    print(f"  {'trial':>5s} {'MPJPE':>7s} {'MPVPE':>7s} {'min':>5s}  " + " ".join(f"{k:>14s}" for k in KEYS))
    for r in sorted(trials, key=lambda r: r["mpjpe"])[:5]:
        mark = "*" if r["chosen"] else " "
        print(f" {mark}{r['trial']:5d} {r['mpjpe']:7.2f} {r['mpvpe']:7.2f} {r['minutes']:5.1f}  "
              + " ".join(f"{fmt(r['params'].get(k, '-')):>14s}" for k in KEYS))

## 2. The course of the studies

The validation MPJPE of every trial (points, in the order Optuna ran them) and the best so far (line), per noise level; the isotropic study in red, the per-joint-noise one in pink. The first trials are fixed starting points (the hand sweep and DPoser's settings, see the configs); TPE takes over after them. Dashed: DPoser and PoseD-Flow on the same validation sequences. A line that is still falling at the last trial means the study stopped early.

In [ ]:
def reference(label, sigma, key="mpjpe"):
    r = R.find(records, label, sigma, "valid")
    return None if r is None else r[key]


with plt.style.context("default"):
    fig, axes = plt.subplots(1, len(SIGMAS), figsize=(6.5 * len(SIGMAS), 4.2))
    for ax, sigma in zip(np.atleast_1d(axes), SIGMAS):
        for name, trials in sorted(studies.items()):
            if trials[0]["sigma"] != sigma:
                continue
            color = FAMILIES[trials[0]["family"]]
            x = [r["trial"] for r in trials]
            y = np.array([r["mpjpe"] for r in trials])
            ax.plot(x, y, "o", ms=4, alpha=0.45, color=color)
            ax.plot(x, np.minimum.accumulate(y), "-", lw=2, color=color, label=f"{name} (best {y.min():.2f})")
            for r in trials:
                if r["chosen"]:
                    ax.plot(r["trial"], r["mpjpe"], "*", ms=15, color=color, markeredgecolor="black", zorder=4)
        for label, short in REFERENCES.items():
            v = reference(label, sigma)
            if v is not None:
                ax.axhline(v, ls="--", lw=1, color=FAMILIES[short], label=f"{short} ({v:.2f})")
        best = min(r["mpjpe"] for r in records if r["sigma"] == sigma and "study" in r)
        ax.set_ylim(best - 0.5, best + 6)
        ax.set_title(f"validation, {round(sigma * 1000)} mm", fontsize=10)
        ax.set_xlabel("trial")
        ax.set_ylabel("MPJPE (mm)")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7, frameon=False)
    plt.tight_layout()
    plt.show()

## 3. The parameters

For one study at a time (selector): the validation MPJPE of every trial against each of its parameters (log axes where the search was on a log scale), the colour the order of the trial (light: early, dark: late), the chosen trial starred. A clear valley means the parameter matters and the range holds the best; a best value at the edge of the range means the range was too narrow; a flat cloud means the parameter hardly matters. The last panel shows MPVPE against MPJPE for the same trials: what the objective did not look at.

In [ ]:
from matplotlib.ticker import FormatStrFormatter, NullFormatter

LOG = {"t_max", "sds_weight", "lr"}


def parameter_figure(name):
    trials = studies[name]
    keys = [k for k in KEYS if k in trials[0]["params"]]
    with plt.style.context("default"):
        fig, axes = plt.subplots(1, len(keys) + 1, figsize=(3.3 * (len(keys) + 1), 3.4), sharey=False)
        order = np.array([r["trial"] for r in trials])
        y = np.array([r["mpjpe"] for r in trials])
        top = np.quantile(y, 0.8)  # the worst fifth is drawn at the top edge, so that the valley is visible
        for ax, k in zip(axes, keys):
            x = np.array([r["params"][k] for r in trials], float)
            ax.scatter(x, np.minimum(y, top), c=order, cmap="Reds", vmin=-len(order) * 0.3, vmax=len(order),
                       s=28, edgecolors="0.3", linewidths=0.4)
            for r in trials:
                if r["chosen"]:
                    ax.plot(r["params"][k], r["mpjpe"], "*", ms=14, color="gold", markeredgecolor="black", zorder=4)
            if k in LOG:
                ax.set_xscale("log")  # four labelled ticks over the range tried, none of the minor ones
                ax.set_xticks(np.geomspace(x.min(), x.max(), 4))
                ax.xaxis.set_major_formatter(FormatStrFormatter("%.2g"))
                ax.xaxis.set_minor_formatter(NullFormatter())
            ax.set_xlabel(k)
            ax.grid(alpha=0.3)
        axes[0].set_ylabel(f"MPJPE (mm, capped at {top:.1f})")
        ax = axes[-1]
        ax.scatter([r["mpvpe"] for r in trials], np.minimum(y, top), c=order, cmap="Reds",
                   vmin=-len(order) * 0.3, vmax=len(order), s=28, edgecolors="0.3", linewidths=0.4)
        for r in trials:
            if r["chosen"]:
                ax.plot(r["mpvpe"], r["mpjpe"], "*", ms=14, color="gold", markeredgecolor="black", zorder=4)
        ax.set_xlabel("MPVPE (mm)")
        ax.grid(alpha=0.3)
        first = trials[0]
        fig.suptitle(f"{name}: {Path(first['model']).parent.name}, {round(first['sigma'] * 1000)} mm, "
                     f"{len(trials)} trials", fontsize=10)
        plt.tight_layout()
    return fig


study_box = widgets.ToggleButtons(options=sorted(studies, key=lambda n: (studies[n][0]["sigma"], n)),
                                  description="study")
out3 = widgets.Output()


def draw3(change=None):
    with out3:
        clear_output(wait=True)
        parameter_figure(study_box.value)
        plt.show()


study_box.observe(draw3, "value")
display(widgets.VBox([study_box, out3]))
draw3()

## 4. All settings: MPJPE against MPVPE and against the cost


Every record of a split and a noise level (selectors), each family in its colour (circle: DPoser's optimization / SDS, square: D-Flow): the trials of the studies and the rows of the caches. The chosen settings are starred. No fronts are drawn: the choice was by MPJPE alone, and the plots only show where the other settings lie.

- **MPJPE against MPVPE**: their table reports MPJPE; MPVPE also sees the twists of the limbs between the joints. A setting can buy MPJPE with MPVPE (more SDS steps did, the sweep of N, 2026-11-17).
- **MPJPE against the network calls** per frame (every step calls the network on every frame): SDS once per step, D-Flow its whole ODE (100 Euler steps) at every step (counted forward only; the backward pass through it comes on top); without a prior none (drawn at 1).

On the test split, their numbers (Tab. 3, MPJPE only) are dashed lines.

In [ ]:
PANELS = [("mpvpe", "mpjpe"), ("nfe", "mpjpe")]
LABELS = {"mpjpe": "MPJPE (mm)", "mpvpe": "MPVPE (mm)", "nfe": "network calls per frame (log; none: 1)"}


def x_of(r, x):
    return r["nfe_plot"] if x == "nfe" else r[x]


def scatter(split, sigma, models):
    rows = [r for r in records if r["split"] == split and r["sigma"] == sigma and r["model"] in models]
    plain = [r for r in records if r["split"] == split and r["sigma"] == sigma and r["model"] == "none"]
    limits = {m: ZOOM * plain[0][m] for m in ("mpjpe", "mpvpe")} if plain else {}
    with plt.style.context("default"):
        fig, axes = plt.subplots(1, len(PANELS), figsize=(7 * len(PANELS), 5.2))
        axes = np.atleast_1d(axes)
        for ax, (x, y) in zip(axes, PANELS):
            seen = [r for r in rows if all(r[m] <= limits.get(m, np.inf) for m in (x, y) if m != "nfe")]
            for family, color in FAMILIES.items():
                mine = [r for r in seen if r["family"] == family]
                for method, marker in MARKERS.items():
                    pts = [r for r in mine if r["method"] == method]
                    if pts:
                        ax.plot([x_of(r, x) for r in pts], [r[y] for r in pts], marker, ls="", ms=7, alpha=0.55,
                                color=color, markeredgecolor="white", markeredgewidth=0.5,
                                label=f"{family}, {'D-Flow' if method == 'dflow' else 'SDS / DPoser'} ({len(pts)})")
                for r in (r for r in mine if r["chosen"]):
                    ax.plot(x_of(r, x), r[y], "*", ms=16, color=color, markeredgecolor="black", zorder=4)
            if split == "test" and y == "mpjpe":
                column = SIGMAS.index(sigma)
                for name, values in PAPER.items():
                    ax.axhline(values[column], color="0.6", lw=0.8, ls="--", zorder=0)
                    if ax is axes[-1]:
                        ax.text(1.0, values[column], f" {name} (paper)", transform=ax.get_yaxis_transform(),
                                fontsize=7, va="center", color="0.4")
            if x == "nfe":
                ax.set_xscale("log")
            ax.set_xlabel(LABELS[x])
            ax.set_ylabel(LABELS[y])
            ax.grid(alpha=0.3)
        axes[0].legend(fontsize=7, frameon=False)
        fig.suptitle(f"motion denoising, {split}, noise {round(sigma * 1000)} mm (star: chosen by validation MPJPE)",
                     fontsize=11)
        plt.tight_layout()
    return fig


split_box = widgets.ToggleButtons(options=["valid", "test"], value="valid", description="split")
sigma_box = widgets.ToggleButtons(options=[("40 mm", 0.04), ("100 mm", 0.1)], value=0.1, description="noise")
models_box = widgets.SelectMultiple(description="models", rows=8, layout=widgets.Layout(width="760px"))
file_box = widgets.Text(value="denoising_settings.pdf", description="file")
save_button = widgets.Button(description="save")
out4 = widgets.Output()
state = {}


def models_of(split, sigma):
    return sorted({r["model"] for r in records if r["split"] == split and r["sigma"] == sigma})


def refresh(change=None):
    shown = models_of(split_box.value, sigma_box.value)
    models_box.options = shown
    models_box.value = tuple(shown)
    draw4()


def draw4(change=None):
    with out4:
        clear_output(wait=True)
        state["fig"] = scatter(split_box.value, sigma_box.value, set(models_box.value))
        plt.show()


def save(_=None):
    folder = Path(SAVE_DIR).expanduser()
    folder.mkdir(parents=True, exist_ok=True)
    state["fig"].savefig(folder / file_box.value, dpi=200, bbox_inches="tight")
    with out4:
        print(f"saved {folder / file_box.value}")


for box in (split_box, sigma_box):
    box.observe(refresh, "value")
models_box.observe(draw4, "value")
save_button.on_click(save)
display(widgets.VBox([widgets.HBox([split_box, sigma_box]), models_box, widgets.HBox([file_box, save_button]), out4]))
refresh()

## 5. Reading

To be completed with the test of the tuned rows (`test2.log`, 2026-11-20). On validation:

- At 40 mm the isotropic study found its best (18.69) late, at trial 36, with 3 x 67 steps; 3 x 133 steps were tried only twice. At 100 mm it found 30.87 at trial 29, with 3 x 133 steps; 3 x 267 only once.
- At 100 mm the best t_min_fraction (0.88) is at the top of its range (0.9): t_min close to t_max, a nearly constant noise level, may be better still. Not tried.
- The per-joint-noise studies did not get below their first point at 40 mm and gained 0.1 mm at 100 mm (2026-11-18 report).
- RSGM is below DPoser and PoseD-Flow in MPJPE at both noise levels; in MPVPE PoseD-Flow is lower at 40 mm (37.2 against about 38).